In [0]:
# Databricks Notebook – LLM Daily Business Summary
# Schedule  : Daily at 07:00 (Databricks Jobs > Schedule)
# Cluster   : Any cluster with access to the catalog/schema
# Widgets   : groq_api_key, slack_webhook_url (set in job parameters)
# pip install: groq (add to cluster init or %pip cell below)

%pip install groq
dbutils.library.restartPython()

import json
import re
from datetime import date, timedelta
from groq import Groq

# ============================================================
# 0. CONFIGURATION  – fill via Databricks Widgets / Secrets
# ============================================================

# Recommended: use Databricks Secrets instead of hardcoding
# GROQ_API_KEY      = dbutils.secrets.get(scope="llm", key="groq_api_key")
# SLACK_WEBHOOK_URL = dbutils.secrets.get(scope="notifications", key="slack_webhook")

# Or widget-based (set in Job parameters):
# GROQ_API_KEY      = dbutils.widgets.get("groq_api_key")
# SLACK_WEBHOOK_URL = dbutils.widgets.get("slack_webhook_url")

# ── Temporary: paste keys directly for dev/test ─────────────
import sys

project_path = "/Workspace/Users/myaswanth18@gmail.com/Automobile-Industry-Data-Analytics-Platform"
if project_path not in sys.path:
    sys.path.append(project_path)

from config import GROQ_API_KEY, SLACK_WEBHOOK as SLACK_WEBHOOK_URL

CATALOG  = "workspace"   # updated to match your stored tables
SCHEMA   = "default"     # updated to match your stored tables
MODEL_ID = "llama-3.3-70b-versatile"  # Groq model – fast and capable

TODAY     = date.today()
YESTERDAY = TODAY - timedelta(days=1)

# ============================================================
# 1. FETCH DATA FROM GOLD TABLES
# ============================================================

def read_gold(table_name: str):
    return spark.read.table(f"{CATALOG}.{SCHEMA}.{table_name}")

def safe_val(val, default=0):
    """Return Python scalar, replacing None with default."""
    return default if val is None else val


# ── 1a. Daily comparison (today vs yesterday JSON blobs) ────
daily_comp_df = read_gold("gold_daily_comparison")
daily_comp = {
    row["domain"]: {
        "today":     json.loads(row["today_metrics"])     if row["today_metrics"]     else {},
        "yesterday": json.loads(row["yesterday_metrics"]) if row["yesterday_metrics"] else {}
    }
    for row in daily_comp_df.collect()
}

# ── 1b. Anomaly summary (flagged rows only) ──────────────────
anomaly_df = (
    read_gold("gold_daily_anomaly_summary")
    .filter("claims_anomaly = 1")
)
anomaly_rows = anomaly_df.collect()

# ── 1c. Dealer scorecard – top 5 and bottom 5 by composite score today ──
scorecard_df = (
    read_gold("gold_dealer_scorecard")
    .filter(f"sale_month_label = '{TODAY.strftime('%Y-%m')}'")
    .select("dealer_id", "dealer_name", "dealer_region",
            "units_sold", "total_revenue", "composite_score",
            "performance_tier", "avg_feedback_score")
    .orderBy("composite_score", ascending=False)
)
top5_dealers    = scorecard_df.limit(5).collect()
bottom5_dealers = scorecard_df.orderBy("composite_score").limit(5).collect()

# ── 1d. Warranty anomaly behavioral (existing gold table) ────
wt_anomaly_df = (
    read_gold("gold_warranty_anomaly_behavioral")
    .filter("anomaly_flag = 1")
    .orderBy("z_score", ascending=False)
    .limit(10)
)
wt_anomaly_rows = wt_anomaly_df.collect()

# ── 1e. Discount anomaly behavioral (existing gold table) ────
disc_anomaly_df = (
    read_gold("gold_discount_anomaly_behavioral")
    .filter("anomaly_flag = 1")
    .orderBy("z_score", ascending=False)
    .limit(10)
)
disc_anomaly_rows = disc_anomaly_df.collect()

# ── 1f. Production efficiency – today ────────────────────────
prod_eff_today = (
    read_gold("gold_production_efficiency")
    .filter(f"prod_month_label = '{TODAY.strftime('%Y-%m')}'")
    .agg(
        {"total_units": "sum", "completed_units": "sum",
         "delayed_units": "sum", "avg_production_time_minutes": "avg"}
    )
    .collect()[0]
)


# ============================================================
# 2. BUILD STRUCTURED PROMPT CONTEXT
# ============================================================

def pct_change(today_val, yest_val) -> str:
    """Return a human-readable % change string."""
    t = safe_val(today_val, 0)
    y = safe_val(yest_val, 0)
    if y == 0:
        return "N/A (no prior data)"
    pct = ((t - y) / abs(y)) * 100
    arrow = "▲" if pct > 0 else ("▼" if pct < 0 else "─")
    return f"{arrow} {abs(pct):.1f}%"


def fmt_anomaly_rows(rows, domain: str) -> str:
    if not rows:
        return "  None detected."
    lines = []
    for r in rows:
        z      = safe_val(r["claims_z_score"])
        chg    = safe_val(r["claims_pct_change"])
        entity = r["dealer_name"] or r["dealer_id"] or "Unknown"
        lines.append(
            f"  • {entity}: z-score={z:.2f}, "
            f"{'+'  if chg >= 0 else ''}{chg:.1f}% vs 30-day avg"
        )
    return "\n".join(lines)


# ── Sales section ────────────────────────────────────────────
sd = daily_comp.get("sales", {})
s_today = sd.get("today", {})
s_yest  = sd.get("yesterday", {})

sales_section = f"""
SALES – {TODAY}
  Units Sold      : {safe_val(s_today.get('units_sold'))}  (yesterday: {safe_val(s_yest.get('units_sold'))})  {pct_change(s_today.get('units_sold'), s_yest.get('units_sold'))}
  Gross Revenue   : ₹{safe_val(s_today.get('gross_revenue')):,.0f}  (yesterday: ₹{safe_val(s_yest.get('gross_revenue')):,.0f})  {pct_change(s_today.get('gross_revenue'), s_yest.get('gross_revenue'))}
  Net Revenue     : ₹{safe_val(s_today.get('net_revenue')):,.0f}  (yesterday: ₹{safe_val(s_yest.get('net_revenue')):,.0f})  {pct_change(s_today.get('net_revenue'), s_yest.get('net_revenue'))}
  Avg Discount    : ₹{safe_val(s_today.get('avg_discount')):,.0f}  (yesterday: ₹{safe_val(s_yest.get('avg_discount')):,.0f})  {pct_change(s_today.get('avg_discount'), s_yest.get('avg_discount'))}
  Unique Customers: {safe_val(s_today.get('unique_customers'))}  (yesterday: {safe_val(s_yest.get('unique_customers'))})
  Active Dealers  : {safe_val(s_today.get('active_dealers'))}
"""

# ── Production section ───────────────────────────────────────
pd_data = daily_comp.get("production", {})
p_today = pd_data.get("today", {})
p_yest  = pd_data.get("yesterday", {})

prod_section = f"""
PRODUCTION – {TODAY}
  Total Units     : {safe_val(p_today.get('total_units'))}  (yesterday: {safe_val(p_yest.get('total_units'))})  {pct_change(p_today.get('total_units'), p_yest.get('total_units'))}
  Completion Rate : {safe_val(p_today.get('completion_rate_pct'))}%  (yesterday: {safe_val(p_yest.get('completion_rate_pct'))}%)  {pct_change(p_today.get('completion_rate_pct'), p_yest.get('completion_rate_pct'))}
  Delay Rate      : {safe_val(p_today.get('delay_rate_pct'))}%  (yesterday: {safe_val(p_yest.get('delay_rate_pct'))}%)  {pct_change(p_today.get('delay_rate_pct'), p_yest.get('delay_rate_pct'))}
  Avg Prod Time   : {safe_val(p_today.get('avg_prod_time_min'))} min  (yesterday: {safe_val(p_yest.get('avg_prod_time_min'))} min)
"""

# ── Service section ──────────────────────────────────────────
svd = daily_comp.get("service", {})
sv_today = svd.get("today", {})
sv_yest  = svd.get("yesterday", {})

service_section = f"""
SERVICE – {TODAY}
  Visits          : {safe_val(sv_today.get('service_visits'))}  (yesterday: {safe_val(sv_yest.get('service_visits'))})  {pct_change(sv_today.get('service_visits'), sv_yest.get('service_visits'))}
  Total Revenue   : ₹{safe_val(sv_today.get('total_service_revenue')):,.0f}  (yesterday: ₹{safe_val(sv_yest.get('total_service_revenue')):,.0f})  {pct_change(sv_today.get('total_service_revenue'), sv_yest.get('total_service_revenue'))}
  Avg Feedback    : {safe_val(sv_today.get('avg_feedback'))}/5  (yesterday: {safe_val(sv_yest.get('avg_feedback'))}/5)  {pct_change(sv_today.get('avg_feedback'), sv_yest.get('avg_feedback'))}
  Warranty-backed : {safe_val(sv_today.get('warranty_backed_services'))}
"""

# ── Warranty section ─────────────────────────────────────────
wd = daily_comp.get("warranty", {})
w_today = wd.get("today", {})
w_yest  = wd.get("yesterday", {})

warranty_section = f"""
WARRANTY – {TODAY}
  Claims Filed    : {safe_val(w_today.get('total_claims'))}  (yesterday: {safe_val(w_yest.get('total_claims'))})  {pct_change(w_today.get('total_claims'), w_yest.get('total_claims'))}
  Total Value     : ₹{safe_val(w_today.get('total_claim_value')):,.0f}  (yesterday: ₹{safe_val(w_yest.get('total_claim_value')):,.0f})  {pct_change(w_today.get('total_claim_value'), w_yest.get('total_claim_value'))}
  Approval Rate   : {safe_val(w_today.get('approval_rate_pct'))}%  (yesterday: {safe_val(w_yest.get('approval_rate_pct'))}%)  {pct_change(w_today.get('approval_rate_pct'), w_yest.get('approval_rate_pct'))}
"""

# ── Inventory section ────────────────────────────────────────
inv = daily_comp.get("inventory", {})
inv_today = inv.get("today", {})

inventory_section = f"""
INVENTORY – snapshot as of {TODAY}
  Total Value     : ₹{safe_val(inv_today.get('total_inventory_value')):,.0f}
  % Below Reorder : {safe_val(inv_today.get('pct_below_reorder'))}%
  Distinct Parts  : {safe_val(inv_today.get('distinct_parts'))}
"""

# ── Anomaly sections ─────────────────────────────────────────
sales_anomaly_rows   = [r for r in anomaly_rows if r["domain"] == "sales"]
wt_daily_anom_rows   = [r for r in anomaly_rows if r["domain"] == "warranty"]
svc_anom_rows        = [r for r in anomaly_rows if r["domain"] == "service"]
prod_anom_rows       = [r for r in anomaly_rows if r["domain"] == "production"]

anomaly_section = f"""
ANOMALIES DETECTED TODAY (z-score > 2, vs rolling 30-day baseline)

  Sales Revenue/Volume Anomalies:
{fmt_anomaly_rows(sales_anomaly_rows, 'sales')}

  Warranty Claim Anomalies (daily):
{fmt_anomaly_rows(wt_daily_anom_rows, 'warranty')}

  Service Feedback Anomalies:
{fmt_anomaly_rows(svc_anom_rows, 'service')}

  Production Delay Anomalies (by plant):
{fmt_anomaly_rows(prod_anom_rows, 'production')}

  Warranty Behavioral Anomalies (30-day pattern, existing detection):
{"  None." if not wt_anomaly_rows else chr(10).join(
    f"  • Dealer {r['dealer_id']}: z-score={safe_val(r['z_score']):.2f}" for r in wt_anomaly_rows[:5]
)}

  Discount Behavioral Anomalies (30-day pattern, existing detection):
{"  None." if not disc_anomaly_rows else chr(10).join(
    f"  • Dealer {r['dealer_id']}: z-score={safe_val(r['z_score']):.2f}, {safe_val(r['discount_pct'])*100:.1f}% discount rate" for r in disc_anomaly_rows[:5]
)}
"""

# ── Dealer scorecard section ──────────────────────────────────
def fmt_dealer_rows(rows) -> str:
    lines = []
    for r in rows:
        lines.append(
            f"  • {r['dealer_name']} ({r['dealer_region']}): "
            f"Score={safe_val(r['composite_score'])}, "
            f"Tier={r['performance_tier']}, "
            f"Units={safe_val(r['units_sold'])}, "
            f"Revenue=₹{safe_val(r['total_revenue']):,.0f}, "
            f"Feedback={safe_val(r['avg_feedback_score'])}/5"
        )
    return "\n".join(lines) if lines else "  No data."

dealer_section = f"""
DEALER PERFORMANCE – Top 5 this month:
{fmt_dealer_rows(top5_dealers)}

DEALER PERFORMANCE – Bottom 5 this month:
{fmt_dealer_rows(bottom5_dealers)}
"""


# ============================================================
# 3. LLM PROMPT
# ============================================================

SYSTEM_PROMPT = """You are an expert automotive business analyst embedded in a data platform.
Your job is to write a crisp, insightful daily business summary for senior leadership.

Rules:
- Write in plain English, no jargon.
- Lead with 2-3 sentence executive summary.
- Then cover each domain (Sales, Production, Service, Warranty, Inventory) in a short paragraph.
- Highlight day-over-day trend changes – call out significant rises or drops with the % figure.
- Dedicate a separate " Anomalies" section – explain each flagged anomaly in one sentence,
  why it matters, and what action should be considered.
- Close with a "Recommended Actions" bullet list (max 5 bullets).
- Tone: confident, data-driven, concise. No filler phrases like "It is worth noting".
- Do NOT reproduce raw numbers that are already obvious from the data – synthesise and interpret.
"""

USER_PROMPT = f"""Here is today's ({TODAY}) automobile business data. Generate the daily business summary.

{sales_section}
{prod_section}
{service_section}
{warranty_section}
{inventory_section}
{anomaly_section}
{dealer_section}

Write the summary now.
"""


# ============================================================
# 4. CALL GROQ LLAMA
# ============================================================

client = Groq(api_key=GROQ_API_KEY)

print("Calling Groq Llama...")
chat_response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT}
    ],
    temperature=0.3,
    max_tokens=1500,
)

summary_text = chat_response.choices[0].message.content
print("=== LLM SUMMARY ===")
print(summary_text)


# ============================================================
# 5. FORMAT FOR SLACK (Block Kit)
# ============================================================

def build_slack_payload(summary: str, report_date: date) -> dict:
    header = f" Daily Automobile Business Summary – {report_date.strftime('%A, %d %B %Y')}"

    chunks = []
    current = []
    for line in summary.splitlines():
        current.append(line)
        if len(current) >= 40 or (line.strip().startswith("⚠️") and current):
            chunks.append("\n".join(current))
            current = []
    if current:
        chunks.append("\n".join(current))

    blocks = [
        {
            "type": "header",
            "text": {"type": "plain_text", "text": header, "emoji": True}
        },
        {"type": "divider"}
    ]

    for chunk in chunks:
        if chunk.strip():
            blocks.append({
                "type": "section",
                "text": {"type": "mrkdwn", "text": chunk[:3000]}
            })
            blocks.append({"type": "divider"})

    blocks.append({
        "type": "context",
        "elements": [{
            "type": "mrkdwn",
            "text": (
                f"Generated by Automobile Analytics Platform · "
                f"Groq {MODEL_ID} · "
                f"Data as of {report_date}"
            )
        }]
    })

    return {"blocks": blocks}


slack_payload = build_slack_payload(summary_text, TODAY)


# ============================================================
# 6. SEND TO SLACK
# ============================================================

import urllib.request

def post_to_slack(webhook_url: str, payload: dict) -> bool:
    body = json.dumps(payload).encode("utf-8")
    req  = urllib.request.Request(
        webhook_url,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            status = resp.getcode()
            if status == 200:
                print(f" Slack message sent successfully (HTTP {status})")
                return True
            else:
                print(f" Slack returned HTTP {status}")
                return False
    except Exception as e:
        print(f" Failed to send to Slack: {e}")
        return False


success = post_to_slack(SLACK_WEBHOOK_URL, slack_payload)

if not success:
    print("\n  Slack delivery failed. Summary printed above for manual forwarding.")


# ============================================================
# 7. AUDIT LOG  – write run metadata to Delta
# ============================================================

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType
import datetime

audit_schema = StructType([
    StructField("report_date",        StringType(),   False),
    StructField("model_used",         StringType(),   True),
    StructField("prompt_tokens",      StringType(),   True),
    StructField("completion_tokens",  StringType(),   True),
    StructField("slack_delivered",    BooleanType(),  True),
    StructField("run_ts",             StringType(),   True),
])

usage = chat_response.usage
audit_row = Row(
    report_date       = str(TODAY),
    model_used        = MODEL_ID,
    prompt_tokens     = str(usage.prompt_tokens),
    completion_tokens = str(usage.completion_tokens),
    slack_delivered   = success,
    run_ts            = datetime.datetime.utcnow().isoformat()
)

audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
(
    audit_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.llm_summary_audit_log")
)

print(f"\n Audit log written to {CATALOG}.{SCHEMA}.llm_summary_audit_log")
print(f"   Prompt tokens: {usage.prompt_tokens}  |  Completion tokens: {usage.completion_tokens}")
